# 1. Data Cleaning

**What this notebook does:** turns the raw competition files (`train.csv`, `test.csv`) into tables with no inconsistent labels and no gaps.

**Why we clean at all**
- Most models cannot handle missing values (they crash or silently drop rows).
- One category written in several ways (`LF`, `low fat`, `Low Fat`) is treated as several categories, which splits one group into small pieces.
- The test file has the same problems, so it must be cleaned **with exactly the same function** as the training file. Otherwise the model would see different data at prediction time than at training time.

**The rule we follow:** change only what is clearly wrong, copy the value from the closest source we have (the same item, the same store), and keep a simple fallback for the few rows where that source does not exist. Under every rule below you will find *why* we chose it.

**One honest note:** the EDA notebook (`2. eda_analysis.ipynb`) later shows that the final model only needs three columns, and none of the columns we fill in here (weight, visibility, size) is among them. We still clean everything, because it is the EDA that proves those columns are not needed, and that proof needs complete, consistent data.

In [1]:
import pandas as pd

# Step 1: Directly load train and test datasets by name
train_path = 'train.csv'
test_path = 'test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Loaded train dataset: {train.shape[0]} rows, {train.shape[1]} columns")
print(f"Loaded test dataset: {test.shape[0]} rows, {test.shape[1]} columns")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

**What this cell does:** reads the two raw files and prints their sizes.

- The training file has 12 columns and 8,523 rows. The test file has 11 columns and 5,681 rows. The missing column in the test file is `Item_Outlet_Sales`, the sales we are asked to predict.
- Both files are loaded here so that the same cleaning can be applied to both.
- (The saved output of this cell is an error because the notebook was saved before the files were uploaded. Run it with `train.csv` and `test.csv` next to the notebook and it prints the sizes above.)

In [ ]:
# Step 2: Inspect Categorical Inconsistencies and Missing Values
print("Train Item_Fat_Content values:\n", train['Item_Fat_Content'].value_counts())
print("\nTrain Outlet_Size values:\n", train['Outlet_Size'].value_counts(dropna=False))

Train Item_Fat_Content values:
 Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

Train Outlet_Size values:
 Outlet_Size
Medium    2793
NaN       2410
Small     2388
High       932
Name: count, dtype: int64


**What the output shows, and what it tells us to fix**

| Finding | Evidence | Why it is a problem |
|---|---|---|
| `Item_Fat_Content` has 5 labels for 2 categories | `Low Fat` 5,089, `Regular` 2,889, but also `LF` 316, `reg` 117, `low fat` 112 | A model would treat `LF` and `Low Fat` as different things |
| `Outlet_Size` is missing for 2,410 training rows | `NaN` 2,410 of 8,523 (28%) | Missing values break most models |
| `Item_Weight` also has gaps | 1,463 missing in train, 976 in test | Same |
| `Item_Visibility` contains zeros | a product on a shelf cannot occupy 0% of the shelf | A fake 0 is false information |

Always inspect before cleaning: every rule in the next cell comes directly from something we saw here.

### Step 3: Data Cleaning Pipeline for Machine Learning

1. **Standardizing categories** in `Item_Fat_Content`.
2. **Imputing missing values** in `Item_Weight` (using the mean of each `Item_Identifier`).
3. **Imputing missing values** in `Outlet_Size` (using the mode of the corresponding `Outlet_Type`).
4. **Handling zero values** in `Item_Visibility` (replacing 0 with the mean of that item's visibility).

## Why we clean each column the way we do

The next code cell contains one function, `clean_dataset`, with four steps. Here is the reasoning behind each step.

### Step 1: Fat-content labels (`LF`, `low fat`, `reg`)

- **What we do:** replace `LF` and `low fat` with `Low Fat`, and `reg` with `Regular`.
- **Why:** these are the same two categories typed in different ways (316 + 112 rows are low fat, 117 rows are regular). We use a lookup (`replace`) instead of guessing, so nothing else changes and no row is lost.
- **Result:** 5 labels become 2 (`Low Fat` 5,517 rows, `Regular` 3,006 rows).

### Step 2: Item weight (1,463 missing in train, 976 in test)

- **What we do:** fill a missing weight with the mean weight of the *same item* (`Item_Identifier`) from its other rows. If the item has no weight in any row, use the median of the whole column.
- **Why the item's own mean:** weight is a property of the product, not of the store. Checked on the cleaned data, 1,539 of the 1,559 items have exactly one weight in every row. So another row of the same item is the best possible source, and for these items the mean is just that one value. This is far better than a global average, which would give a heavy bag of rice the weight of a small tin.
- **Why the median as the fallback:** a few items never have a weight, so there is nothing to copy. For those we use a typical value for the whole column. We choose the *median* (the middle value) over the mean because a few very heavy or very light items would drag the mean away from what is typical. This affects only a handful of rows.
- **What we cannot know:** the true weight of those few items. The fallback is a neutral guess, not a measurement.

### Step 3: Item visibility (zeros)

- **What we do:** treat every `0` as "unknown", then fill it with the mean of that item's other (non-zero) visibilities, and fall back to the median of all non-zero visibilities.
- **Why zeros count as missing:** visibility is the share of the store's display space a product gets. A product that is actually sold in a store cannot have 0%, so a zero is a recording gap, not a real measurement. Leaving the zeros would also pull averages and ratios down artificially.
- **Why the item's own mean:** the same item has a similar visibility in different stores (typical spread within one item is about 0.013), while visibility varies a lot between items (spread about 0.050 over the whole table). So the item's own average is a much closer guess than the global one.
- **Why the median as the fallback:** same reason as for weight, it is robust to unusual values.
- **Assumption to be aware of:** the data cannot prove that every zero is a mistake. We assume it, because the alternative (a product taking no space) is physically impossible.

### Step 4: Outlet size (2,410 missing in train)

- **What we do:** fill every missing size with the constant `Small` (the argument `outlet_size_fill`).
- **Why a constant and not "the same item's mean":** size belongs to the *store*, so the gaps are not scattered. The 2,410 missing rows are exactly all the rows of three stores (OUT010 has 555, OUT017 has 926, OUT045 has 929). There is no other row of those stores to copy from.
- **Why `Small`:** among the stores whose size is known, the only known grocery store (OUT019) is `Small`, and among the known Type 1 supermarkets (OUT013 High, OUT035 Small, OUT046 Small, OUT049 Medium) `Small` is the most common. OUT010 is a grocery store and OUT017 and OUT045 are Type 1 supermarkets, so `Small` is the most likely value for all three.
- **A note on the list above:** the summary cell above says "mode of the corresponding `Outlet_Type`". The code uses the constant `Small` instead. Here the two give the *same* answer (the most common size is `Small` for both store types), so nothing changes, but the code is what actually runs.
- **How much it matters:** very little. This is an educated guess, and the EDA shows that store size adds nothing once we know which store it is.

### Why one function, applied to both files

`clean_dataset` is applied once to `train` and once to `test` with the same settings. Every rule is learned *from the data itself* (an item's own rows, the column median), never from the sales, so there is no leakage of the answers into the features.

In [ ]:
import numpy as np

def clean_dataset(df, outlet_size_fill='Small'):
    df_clean = df.copy()

    # 1. Standardize Item_Fat_Content
    df_clean['Item_Fat_Content'] = df_clean['Item_Fat_Content'].replace({
        'LF': 'Low Fat',
        'low fat': 'Low Fat',
        'reg': 'Regular'
    })

    # 2. Item_Weight

    # mean weight for each item identifier
    item_weight_map = df_clean.groupby('Item_Identifier')['Item_Weight'].transform('mean')
    df_clean['Item_Weight'] = df_clean['Item_Weight'].fillna(item_weight_map)

    # remaining missing weights using the global median
    global_median_weight = df_clean['Item_Weight'].median()
    df_clean['Item_Weight'] = df_clean['Item_Weight'].fillna(global_median_weight)


    # 3. Impute Item_Visibility

    # Replace zero values with NaN first
    df_clean['Item_Visibility'] = df_clean['Item_Visibility'].replace(0, np.nan)

    # Impute missing visibility using the same item's mean nonzero visibility
    item_visibility_map = df_clean.groupby('Item_Identifier')['Item_Visibility'].transform('mean')
    df_clean['Item_Visibility'] = df_clean['Item_Visibility'].fillna(item_visibility_map)

    # Impute any remaining missing visibility values using the global median of nonzero visibility
    global_median_visibility = df_clean['Item_Visibility'].median()
    df_clean['Item_Visibility'] = df_clean['Item_Visibility'].fillna(global_median_visibility)

    # 4. Handle missing Outlet_Size values (using 'Small' or 'Unknown' depending on mode)
    df_clean['Outlet_Size'] = df_clean['Outlet_Size'].fillna(outlet_size_fill)

    return df_clean

# Run the updated cleaning pipeline matching all criteria
train_clean = clean_dataset(train, outlet_size_fill='Small')
test_clean = clean_dataset(test, outlet_size_fill='Small')

# Save clean dataframes to separate new CSV files
train_clean.to_csv('train_cleaned.csv', index=False)
test_clean.to_csv('test_cleaned.csv', index=False)

print("Cleaning complete! Saved as 'train_cleaned.csv' and 'test_cleaned.csv'.\n")
print("Cleaned Train Missing Values:\n", train_clean.isnull().sum())
print("\nCleaned Test Missing Values:\n", test_clean.isnull().sum())

Cleaning complete! Saved as 'train_cleaned.csv' and 'test_cleaned.csv'.

Cleaned Train Missing Values:
 Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

Cleaned Test Missing Values:
 Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
dtype: int64


**What the output shows**

- Every column now has **0 missing values**, in both the training and the test data.
- The cleaned tables are saved as CSV files (`train_cleaned.csv`, `test_cleaned.csv`). In this project they are kept in the main folder as **`train_cl.csv`** and **`test_cl.csv`**, which is what the next notebook and the final model read.

**What cleaning deliberately does not do**
- It does not remove outliers: very high sales are real, and a model that has to predict them should see them.
- It does not create new columns. Which columns are useful is decided in the next notebook, by evidence.

**Next:** `2. eda_analysis.ipynb`.